# Steam Games Market Analysis — ETL Pipeline
**Author:** Mohammed (E-JUST / DEPI Microsoft Data Engineer Track)  
**Engine:** PySpark 3.x on Docker  
**Dataset:** [Steam Games Dataset](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset) (~122,000 records)  

---

### Pipeline Overview
1. **Extract** — Raw CSV ingestion with schema inference disabled
2. **Schema Realignment** — Fix the 13-column index shift caused by a merged `DiscountDLC count` header
3. **Feature Engineering** — Derived metrics (net revenue, mean owners, inferred age rating)
4. **Data Quality** — ASCII name filtering, null coalescing, type enforcement
5. **Export** — Single-partition CSV for local use + JDBC load into PostgreSQL

In [ ]:
"""
Project: Steam Games Market Analysis - ETL Pipeline
Author: Mohammed (E-JUST / DEPI Microsoft Data Engineer Track)
Description: End-to-end Spark pipeline for schema correction, 
             feature engineering, and data integrity validation.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as f

# 1. Spark Session Configuration
# Allocated 4GB to executors to handle repartitioning and math operations efficiently.
spark = SparkSession.builder \
    .appName("Steam_ETL_Production") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# ====================  BRONZE LAYER (Raw Ingestion)  ====================
# Raw data loaded as-is. inferSchema=False to prevent type errors on shifted columns.
df_raw = spark.read.csv("games.csv", header=True, inferSchema=False)
headers = df_raw.columns

# ====================  SILVER LAYER (Cleaned + Validated)  ====================
# 3. Schema Realignment 
# Addressing the index offset caused by the 'DiscountDLC count' header.
df_aligned = df_raw.select(
    *[f.col(headers[i]) for i in range(7)],
    f.col(headers[7]).cast("float").alias("Discount_Percent"),
    f.col(headers[8]).cast("int").alias("DLC_Count"),
    # Re-aligning data from Supported languages (Index 9) to Metacritic URL (Index 20)
    *[f.col(headers[i+1]).alias(headers[i]) for i in range(9, 21)],
    # Skipping dislocated User_Score data (Index 22) to align review metrics
    *[f.col(headers[i+1]).alias(headers[i]) for i in range(22, 38)]
)

# ====================  GOLD LAYER (Business-Ready Metrics)  ====================
# 4. Feature Engineering and Type Optimization
df_processed = df_aligned.withColumn(
    "Release_Date", f.to_date(f.col("Release date"), "yyyy-MM-dd")
).withColumn(
    # OS Support optimization: Converting strings to Integers (1/0) for Power BI performance
    "Windows", f.col("Windows").cast("int")
).withColumn(
    "Mac", f.col("Mac").cast("int")
).withColumn(
    "Linux", f.col("Linux").cast("int")
).withColumn(
    # Financial Analytics: Calculating Net Revenue (70% after platform fees)
    "mean_owners", (
        f.split(f.col("Estimated owners"), " - ").getItem(0).cast("int") + 
        f.split(f.col("Estimated owners"), " - ").getItem(1).cast("int")
    ) / 2
).withColumn(
    "estimated_net_revenue", f.round(
        (f.col("Price").cast("float") * (1 - (f.col("Discount_Percent") / 100)) * f.col("mean_owners")) * 0.7, 2
    )
).withColumn(
    # Content Metadata: Age Inference from Tags
    "Inferred_Required_Age",
    f.when(f.col("Tags").rlike("(?i)Mature|Gore|Nudity|Sexual Content|Violent"), 18)
     .when(f.col("Tags").rlike("(?i)Teen|Blood"), 13)
     .otherwise(f.col("Required age").cast("int"))
)

# 5. Data Quality Filtering
df_final = df_processed.filter(
    f.col("Name").rlike("^[\\x00-\\x7F]+$")
).drop("About the game", "Required age", "User score", "Movies", "Release date")

# 6. Integrity Check (Validation Suite)
# This calculates null counts for every column to detect if the 'shift' left gaps.
print("--- Data Integrity Profile ---")
null_counts = df_final.select([f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_final.columns])
null_counts.show(vertical=True)

# 7. Final Export
# Repartition(1) ensures a single CSV file for easier local management.
df_final.repartition(1).write.csv("refined_steam_data", header=True, mode="overwrite")

In [ ]:
"""
Steam Data Warehouse: Final Migration (Production Grade)
Author: Mohammed (E-JUST / DEPI Microsoft Data Engineer Track)
Description: Final type-enforcement and schema-alignment for 122k records.
"""

from pyspark.sql import functions as f

# --- 1. THE HARD CASTING LAYER ---
# We are forcing every 'ABC' column into its '123' (Integer) or 'Date' identity.
df_final = df_aligned.withColumn(
    # Date Handling: MMM d, yyyy -> Date Type
    "temp_date", f.to_date(f.col("Release date"), "MMM d, yyyy")
).withColumn(
    # 2. Convert to the Standard SQL Format (YYYY-MM-DD)
    "Parsed_Date", f.date_format(f.col("temp_date"), "yyyy-MM-dd").cast("date")
).withColumn(
    # Cast review metrics and scores to Integer
    "Peak_CCU", f.col("Peak CCU").cast("int")
).withColumn(
    "Positive", f.col("Positive").cast("int")
).withColumn(
    "Negative", f.col("Negative").cast("int")
).withColumn(
    "Achievements", f.col("Achievements").cast("int")
).withColumn(
    "Metacritic_score", f.col("Metacritic score").cast("int")
).withColumn(
    # Boolean Enforcement: Explicitly checking for truthy values to avoid nulls
    "Windows", f.when(f.col("Windows").rlike("(?i)true|1|yes"), True).otherwise(False)
).withColumn(
    "Mac", f.when(f.col("Mac").rlike("(?i)true|1|yes"), True).otherwise(False)
).withColumn(
    "Linux", f.when(f.col("Linux").rlike("(?i)true|1|yes"), True).otherwise(False)
).withColumn(
    "Price", f.col("Price").cast("float")
).withColumn(
    "mean_owners", (
        f.split(f.col("Estimated owners"), " - ").getItem(0).cast("int") + 
        f.split(f.col("Estimated owners"), " - ").getItem(1).cast("int")
    ) / 2
).withColumn(
    "estimated_net_revenue", f.round(
        (f.col("Price") * (1 - (f.col("Discount_Percent").cast("float") / 100)) * f.col("mean_owners")) * 0.7, 2
    )
).drop("Release date", "About the game", "Required age", "User score", "Movies", "Metacritic score")

# --- 2. FINAL SPARK CHECK ---
# If this says 'string' for a number column, we haven't fixed it yet.
# Explicitly casting for the SQL 'Contract'
df_final = df_aligned.withColumn("Parsed_Date", f.to_date(f.col("Release date"), "MMM d, yyyy")) \
                     .withColumn("Peak_CCU", f.col("Peak CCU").cast("int")) \
                     .withColumn("Positive", f.col("Positive").cast("int")) \
                     .withColumn("Negative", f.col("Negative").cast("int")) \
                     .withColumn("Windows", f.col("Windows").cast("boolean")) \
                     .withColumn("Price", f.col("Price").cast("float"))

# --- 3. THE JDBC HANDSHAKE ---
jdbc_url = "jdbc:postgresql://postgres_general:5432/postgres"
db_props = {"user": "admin", "password": "admin", "driver": "org.postgresql.Driver"}
df_final.repartition(1).write.csv("refined_steam_data", header=True, mode="overwrite")
df_final.write.jdbc(url=jdbc_url, table="fact_steam_games", mode="overwrite", properties=db_props)